# Métricas de evaluación para modelos de clasificación 📊

En este notebook aprenderemos a evaluar un clasificador más allá de *accuracy*. Veremos:

- matriz de confusión, accuracy, precision, recall, especificidad y F1;
- probabilidades y **threshold (umbral) de decisión**;
- curvas ROC y Precision–Recall;
- ROC-AUC y Average Precision (PR-AUC);
- cómo elegir una métrica y un threshold según el costo del error.

> Idea central: no existe una métrica universalmente mejor. La métrica correcta depende del problema y del costo de los falsos positivos y falsos negativos.

## 1. El problema de usar solamente accuracy

Imaginemos un modelo que predice si tendremos un accidente automovilístico hoy. Si una persona conduce durante 55 años, tiene aproximadamente $55\times365=20{,}075$ días posibles de manejo, pero solo unos cuantos tendrán un accidente.

Un programa que siempre responda **“hoy no tendrás un accidente”** acertaría casi todos los días. Tendría un accuracy excelente, pero sería inútil para detectar accidentes. A esto se le llama un problema de **clases desbalanceadas**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42

## 2. Datos y modelo de ejemplo

Crearemos datos sintéticos donde solo cerca del 5% de los casos son positivos. Usamos una separación entrenamiento/prueba **estratificada** para conservar esa proporción.

In [ ]:
X, y = make_classification(
    n_samples=5000, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.95, 0.05], class_sep=1.1, flip_y=0.01,
    random_state=RANDOM_STATE
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pd.Series(y_test).value_counts(normalize=True).rename("proporción")

### Baseline: predecir siempre la clase mayoritaria

Una métrica debe compararse con una referencia sencilla. Aquí la referencia predice siempre 0.

In [ ]:
y_baseline = np.zeros_like(y_test)
print(f"Accuracy del baseline: {accuracy_score(y_test, y_baseline):.3f}")
print(f"Recall del baseline:   {recall_score(y_test, y_baseline):.3f}")

## 3. Matriz de confusión

Para una clase positiva (1) y una negativa (0):

| | Predicción positiva | Predicción negativa |
|---|---:|---:|
| **Real positiva** | TP: verdadero positivo | FN: falso negativo |
| **Real negativa** | FP: falso positivo | TN: verdadero negativo |

La matriz permite ver **qué tipo de error** comete el modelo, no solamente cuántos.

In [ ]:
# predict() usa por defecto un threshold de 0.5 en este clasificador.
y_pred = model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de confusión (threshold = 0.5)")
plt.show()

## 4. Métricas derivadas

### Accuracy

$$\text{accuracy}=\frac{TP+TN}{TP+TN+FP+FN}$$

Es fácil de interpretar, pero puede ocultar un mal desempeño en la clase minoritaria.

### Precision

De todos los casos predichos como positivos, ¿cuántos sí lo eran?

$$\text{precision}=\frac{TP}{TP+FP}$$

Es importante cuando los **falsos positivos** son costosos. Por ejemplo, bloquear una transacción legítima.

### Recall o sensibilidad

De todos los positivos reales, ¿cuántos detectamos?

$$\text{recall}=\frac{TP}{TP+FN}$$

Es importante cuando los **falsos negativos** son costosos. Por ejemplo, no detectar una enfermedad.

### Especificidad

De todos los negativos reales, ¿cuántos reconocimos como negativos?

$$\text{specificity}=\frac{TN}{TN+FP}=1-\text{FPR}$$

### F1 y F-beta

F1 es la media armónica de precision y recall:

$$F_1=2\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$$

$F_\beta$ permite dar más peso al recall ($\beta>1$) o a precision ($\beta<1$).

In [ ]:
def classification_metrics(y_true, y_pred):
    # ravel() devuelve TN, FP, FN, TP en ese orden para clasificación binaria.
    from sklearn.metrics import confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall/sensibilidad": recall_score(y_true, y_pred),
        "especificidad": tn / (tn + fp),
        "F1": f1_score(y_true, y_pred),
        "F2": fbeta_score(y_true, y_pred, beta=2),
    }

pd.Series(classification_metrics(y_test, y_pred)).round(3)

In [ ]:
print(classification_report(y_test, y_pred, target_names=["negativo", "positivo"]))

## 5. Probabilidad y threshold de decisión

Muchos clasificadores generan un **score** o una probabilidad estimada. Después se aplica un umbral:

$$\hat y = 1 \quad \text{si} \quad P(y=1\mid x)\geq t$$

El threshold $t=0.5$ es común, pero no necesariamente óptimo.

- Bajar el threshold suele detectar más positivos: sube recall, pero aparecen más falsos positivos.
- Subirlo exige mayor confianza: suele subir precision, pero aparecen más falsos negativos.

> El threshold debe elegirse con datos de validación, nunca optimizarse directamente sobre el conjunto de prueba. Aquí usamos test únicamente con fines didácticos.

In [ ]:
y_score = model.predict_proba(X_test)[:, 1]

def predict_with_threshold(scores, threshold):
    return (scores >= threshold).astype(int)

thresholds_to_try = [0.10, 0.30, 0.50, 0.70]
comparison = pd.DataFrame({
    t: classification_metrics(y_test, predict_with_threshold(y_score, t))
    for t in thresholds_to_try
}).T
comparison.index.name = "threshold"
comparison.round(3)

In [ ]:
threshold_grid = np.linspace(0.01, 0.99, 99)
rows = []
for t in threshold_grid:
    pred_t = predict_with_threshold(y_score, t)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t),
        "F1": f1_score(y_test, pred_t),
    })
threshold_df = pd.DataFrame(rows)

threshold_df.plot(x="threshold", y=["precision", "recall", "F1"], figsize=(9, 5))
plt.ylim(0, 1.02)
plt.ylabel("valor de la métrica")
plt.title("Efecto del threshold")
plt.show()

### Dos maneras de seleccionar un threshold

1. **Optimizar una métrica**, por ejemplo F1.
2. **Cumplir una restricción de negocio**, por ejemplo recall $\geq 0.80$, y dentro de los thresholds que la cumplen elegir el de mayor precision.

En aplicaciones reales también puede minimizarse un costo explícito: $C_{FP}\cdot FP+C_{FN}\cdot FN$.

In [ ]:
best_f1_row = threshold_df.loc[threshold_df["F1"].idxmax()]
print("Threshold que maximiza F1 en estos datos:")
display(best_f1_row.round(3))

candidates = threshold_df[threshold_df["recall"] >= 0.80]
if not candidates.empty:
    chosen = candidates.loc[candidates["precision"].idxmax()]
    print("Mejor precision manteniendo recall >= 0.80:")
    display(chosen.round(3))
else:
    print("Ningún threshold evaluado alcanza recall >= 0.80")

## 6. Curva ROC y ROC-AUC

La curva ROC muestra el desempeño para **todos los thresholds**:

- eje Y: TPR = recall = $TP/(TP+FN)$;
- eje X: FPR = $FP/(FP+TN)=1-\text{especificidad}$.

**ROC-AUC** es el área bajo esa curva. También puede interpretarse como la probabilidad de que el modelo asigne un score mayor a un positivo elegido al azar que a un negativo elegido al azar.

- 0.5: ranking similar al azar;
- 1.0: separación perfecta;
- menor que 0.5: el ranking está, en general, invertido.

ROC-AUC mide la capacidad de **ranking** y no fija un threshold. Además, con clases muy desbalanceadas puede parecer optimista porque abundan los verdaderos negativos.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_score)
roc_auc = roc_auc_score(y_test, y_score)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Modelo (ROC-AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar")
plt.xlabel("False Positive Rate (1 - especificidad)")
plt.ylabel("True Positive Rate (recall)")
plt.title("Curva ROC")
plt.legend()
plt.show()

## 7. Curva Precision–Recall y PR-AUC

La curva Precision–Recall se enfoca en el desempeño sobre la clase positiva y suele ser más informativa cuando esta clase es rara.

Aquí resumimos la curva con **Average Precision (AP)**, una forma escalonada de calcular el área PR. La referencia de un clasificador aleatorio es aproximadamente la prevalencia de la clase positiva, no 0.5.

> Conviene indicar qué implementación se usa al reportar “PR-AUC”, ya que la interpolación trapezoidal y Average Precision no siempre producen el mismo valor.

In [ ]:
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_score)
ap = average_precision_score(y_test, y_score)
prevalence = y_test.mean()

plt.figure(figsize=(7, 6))
plt.plot(pr_recall, pr_precision, label=f"Modelo (AP = {ap:.3f})")
plt.axhline(prevalence, linestyle="--", color="gray",
            label=f"Baseline = prevalencia ({prevalence:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision–Recall")
plt.legend()
plt.show()

In [ ]:
pd.Series({
    "ROC-AUC": roc_auc,
    "Average Precision (PR)": ap,
    "prevalencia positiva": prevalence,
}).round(3)

## 8. Caso práctico: reconocer el dígito 5 ✍️

Ahora repetiremos el flujo completo con imágenes de dígitos, siguiendo la progresión didáctica del capítulo de clasificación de *Hands-On Machine Learning*. Usaremos `load_digits`, que viene incluido con scikit-learn y no necesita una descarga.

Cada observación es una imagen de $8\times8$ pixeles. Primero convertiremos el problema multiclase en uno binario: **5** contra **no 5**. Después obtendremos predicciones *out-of-fold* con validación cruzada, para no evaluar cada ejemplo con un modelo que ya lo vio durante su entrenamiento.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import (
    StratifiedKFold, cross_val_predict, cross_val_score
)

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for image, label, ax in zip(digits.images[:10], y_digits[:10], axes.ravel()):
    ax.imshow(image, cmap="binary")
    ax.set_title(f"Etiqueta: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Forma de X: {X_digits.shape}")

### 8.1 Clasificación binaria y una baseline honesta

Al etiquetar solo los cincos como positivos, cerca del 90% de las observaciones serán negativas. Por eso comparamos el modelo con `DummyClassifier`, que ignora las imágenes y predice según una estrategia trivial.

In [ ]:
y_is_5 = (y_digits == 5)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

sgd_5 = SGDClassifier(loss="log_loss", random_state=RANDOM_STATE)
dummy_5 = DummyClassifier(strategy="most_frequent")

cv_accuracy = pd.DataFrame({
    "SGD": cross_val_score(sgd_5, X_digits, y_is_5, cv=cv, scoring="accuracy"),
    "baseline": cross_val_score(dummy_5, X_digits, y_is_5, cv=cv, scoring="accuracy"),
})
display(cv_accuracy.round(3))
display(cv_accuracy.agg(["mean", "std"]).round(3))
print(f"Prevalencia de cincos: {y_is_5.mean():.1%}")

Un accuracy alto del baseline no significa que reconozca cincos: puede conseguirlo prediciendo siempre “no es 5”. Para calcular precision y recall sin fuga de información usamos `cross_val_predict`: cada predicción procede de un modelo entrenado con otros folds.

In [ ]:
y_pred_5 = cross_val_predict(sgd_5, X_digits, y_is_5, cv=cv)

ConfusionMatrixDisplay.from_predictions(
    y_is_5, y_pred_5, display_labels=["no es 5", "es 5"], cmap="Blues"
)
plt.title("Predicciones out-of-fold")
plt.show()

pd.Series(classification_metrics(y_is_5, y_pred_5)).round(3)

### 8.2 El score de decisión y el threshold

`SGDClassifier` produce un score mediante `decision_function`. El score no es necesariamente una probabilidad: su signo y magnitud expresan de qué lado, y a qué distancia, se encuentra una observación respecto a la frontera de decisión. Por defecto, SGD predice la clase positiva cuando el score es mayor o igual que cero.

In [ ]:
scores_5 = cross_val_predict(
    sgd_5, X_digits, y_is_5, cv=cv, method="decision_function"
)
precisions_5, recalls_5, thresholds_5 = precision_recall_curve(y_is_5, scores_5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(thresholds_5, precisions_5[:-1], "--", label="Precision")
axes[0].plot(thresholds_5, recalls_5[:-1], label="Recall")
axes[0].axvline(0, color="black", linestyle=":", label="threshold por defecto")
axes[0].set(xlabel="threshold", ylabel="métrica", ylim=(0, 1.02),
            title="Precision y recall vs. threshold")
axes[0].legend()

axes[1].plot(recalls_5, precisions_5)
axes[1].set(xlabel="recall", ylabel="precision", xlim=(0, 1), ylim=(0, 1.02),
            title="Curva Precision–Recall")
plt.tight_layout()
plt.show()

### 8.3 Elegir un threshold para una precision objetivo

Supongamos que solo queremos marcar un dígito como 5 cuando el modelo alcance al menos 90% de precision. Entre todos los puntos que cumplen la restricción, elegimos el de mayor recall. Esta forma explícita evita depender de la posición u orden particular de los arrays.

In [ ]:
target_precision = 0.90
# El último valor de precision/recall no tiene un threshold asociado.
valid = np.flatnonzero(precisions_5[:-1] >= target_precision)
if valid.size:
    best_idx = valid[np.argmax(recalls_5[:-1][valid])]
    threshold_90 = thresholds_5[best_idx]
    y_pred_90 = scores_5 >= threshold_90
    print(f"Threshold elegido: {threshold_90:.3f}")
    print(f"Precision obtenida: {precision_score(y_is_5, y_pred_90):.3f}")
    print(f"Recall obtenido:    {recall_score(y_is_5, y_pred_90):.3f}")
else:
    print("El modelo no alcanza la precision objetivo.")

### 8.4 Comparar modelos con ROC-AUC y Average Precision

Compararemos SGD con Random Forest usando scores *out-of-fold*. Para SGD usamos la función de decisión; para Random Forest usamos la probabilidad estimada de la clase positiva. Las escalas no necesitan coincidir: ROC-AUC y AP evalúan principalmente el orden de los casos.

In [ ]:
forest_5 = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
probas_forest_5 = cross_val_predict(
    forest_5, X_digits, y_is_5, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

model_scores = {"SGD": scores_5, "Random Forest": probas_forest_5}
summary = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, scores in model_scores.items():
    fpr_m, tpr_m, _ = roc_curve(y_is_5, scores)
    p_m, r_m, _ = precision_recall_curve(y_is_5, scores)
    roc_m = roc_auc_score(y_is_5, scores)
    ap_m = average_precision_score(y_is_5, scores)
    summary.append({"modelo": name, "ROC-AUC": roc_m, "AP": ap_m})
    axes[0].plot(fpr_m, tpr_m, label=f"{name}: {roc_m:.3f}")
    axes[1].plot(r_m, p_m, label=f"{name}: {ap_m:.3f}")

axes[0].plot([0, 1], [0, 1], "k:", label="azar")
axes[0].set(xlabel="FPR", ylabel="TPR / recall", title="ROC")
axes[1].axhline(y_is_5.mean(), color="black", linestyle=":", label="prevalencia")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision–Recall")
for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(summary).set_index("modelo").round(3)

### 8.5 Volver al problema multiclase

En multiclase, precision, recall y F1 necesitan una estrategia de promedio:

- `macro`: calcula la métrica por clase y da el mismo peso a cada una;
- `weighted`: pondera cada clase por su cantidad de ejemplos;
- `micro`: suma TP, FP y FN globalmente antes de calcular la métrica.

Una matriz de confusión normalizada por fila permite responder: “de todos los dígitos reales de esta clase, ¿en qué clases fueron predichos?”.

In [ ]:
sgd_multi = SGDClassifier(loss="log_loss", random_state=RANDOM_STATE)
y_pred_multi = cross_val_predict(sgd_multi, X_digits, y_digits, cv=cv)

print(f"F1 macro:    {f1_score(y_digits, y_pred_multi, average='macro'):.3f}")
print(f"F1 weighted: {f1_score(y_digits, y_pred_multi, average='weighted'):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_digits, y_pred_multi, ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Conteos")
ConfusionMatrixDisplay.from_predictions(
    y_digits, y_pred_multi, normalize="true", values_format=".0%",
    ax=axes[1], cmap="Blues", colorbar=False
)
axes[1].set_title("Normalizada por clase real")
plt.tight_layout()
plt.show()

### Lo que añade este caso práctico

El primer ejemplo sintético facilita entender las fórmulas. El ejemplo de dígitos muestra un flujo de evaluación más cercano a un proyecto real: baseline, validación cruzada, scores fuera de muestra, selección de threshold, comparación de modelos y análisis multiclase.

> Nota de atribución: esta secuencia está adaptada conceptualmente del notebook de clasificación de Aurélien Géron; el texto, las visualizaciones y el código de esta versión fueron reformulados para esta clase y para el dataset `load_digits`.

## 9. ¿Qué métrica elegir?

| Situación | Métrica o criterio útil |
|---|---|
| Clases balanceadas y errores parecidos | Accuracy |
| Un falso positivo es especialmente costoso | Precision / especificidad |
| Un falso negativo es especialmente costoso | Recall / sensibilidad / $F_\beta$ con $\beta>1$ |
| Se busca equilibrio entre precision y recall | F1 |
| Importa ordenar casos y comparar sin fijar threshold | ROC-AUC |
| La clase positiva es muy rara | Curva PR / Average Precision |
| Las probabilidades alimentan decisiones o costos | Log loss, Brier score y calibración |

Una buena evaluación suele reportar más de una métrica, junto con la matriz de confusión y el threshold usado.

## 10. Buenas prácticas

1. Separar entrenamiento, validación y prueba; usar validación cruzada si los datos son pocos.
2. Elegir hiperparámetros y threshold con validación; reservar test para la evaluación final.
3. Comparar contra un baseline sencillo.
4. Reportar variabilidad (por ejemplo, intervalos de confianza o resultados por fold).
5. Revisar métricas por subgrupos relevantes para detectar fallos sistemáticos.
6. Si se usarán probabilidades, comprobar su **calibración**: un grupo con score 0.8 debería contener cerca de 80% de positivos.
7. Recordar que cambiar el threshold cambia las predicciones, pero no ROC-AUC ni Average Precision, porque estas evalúan el ranking completo.

## 11. Preguntas para discutir o practicar ✍️

1. En detección de tumores, ¿qué error preocupa más: FP o FN? ¿Qué métrica priorizarías?
2. En un filtro automático de spam, ¿qué ocurre si el threshold es demasiado bajo?
3. Modifica `weights` en `make_classification`. ¿Cómo cambian accuracy, ROC-AUC y Average Precision?
4. Encuentra el threshold que minimiza un costo donde un FN cuesta 10 veces más que un FP.
5. ¿Por qué dos modelos con el mismo ROC-AUC pueden ser muy distintos en el threshold que interesa al negocio?

In [ ]:
# Ejercicio 4: completa o modifica los costos.
from sklearn.metrics import confusion_matrix

cost_fp, cost_fn = 1, 10
costs = []
for t in threshold_grid:
    pred_t = predict_with_threshold(y_score, t)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    costs.append({"threshold": t, "FP": fp, "FN": fn,
                  "costo_total": cost_fp * fp + cost_fn * fn})

cost_df = pd.DataFrame(costs)
cost_df.loc[cost_df["costo_total"].idxmin()]